<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Arriendos en Tunja con Modelos Lineales 🏢
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 01
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/01_Modelos_Lineales_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 01: Modelos Lineales Supervisados** ([README del módulo](../01%20-%20Modelos%20Lineales%20Supervisados/README.md)) con un problema **nuevo**: modelar el **arriendo mensual** (en millones de pesos) de apartamentos de una ciudad intermedia a partir de sus características.

Al terminar serás capaz de:

1. **Ajustar una regresión lineal** con `scikit-learn` y reproducirla con la **ecuación normal** en NumPy, y medir su error frente a una línea base.
2. **Diagnosticar multicolinealidad** con el factor de inflación de la varianza (VIF) y mitigarla con **Ridge**.
3. **Convertir** el problema en clasificación con **regresión logística**: probabilidades, umbral de decisión y *odds ratios*.
4. **Seleccionar variables** con la penalización **Lasso** (L1).

> 📚 **Repasa si lo necesitas:** [Fundamentos del Aprendizaje Supervisado](../01%20-%20Modelos%20Lineales%20Supervisados/00_Fundamentos_del_Aprendizaje_Supervisado.ipynb) · [Regresión Lineal](../01%20-%20Modelos%20Lineales%20Supervisados/01_Regresion_Lineal.ipynb) · [Regresión Logística](../01%20-%20Modelos%20Lineales%20Supervisados/02_Regresion_Logistica.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet y sin archivos externos** (datos sintéticos con semilla fija). Bibliotecas: `numpy`, `pandas`, `scikit-learn`.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score, recall_score

warnings.filterwarnings("ignore")


def generar_apartamentos(n=900, seed=11):
    rng = np.random.default_rng(seed)
    area = rng.normal(70, 18, n).clip(30, 150)                       # m² construidos
    area_util = 0.92 * area + rng.normal(0, 1.0, n)                  # m² útiles: casi una copia del área
    habitaciones = np.clip(np.round(area / 28 + rng.normal(0, .5, n)), 1, 5)
    estrato = rng.integers(2, 6, n).astype(float)
    antiguedad = rng.uniform(0, 40, n)                               # años de construido
    distancia = rng.gamma(2.5, 1.2, n)                               # km al centro
    ruido = rng.normal(0, 1, (n, 3))                                 # tres columnas sin relación con el arriendo
    arriendo = (0.35 + 0.022 * area + 0.30 * estrato - 0.012 * antiguedad - 0.10 * distancia
                + 0.08 * habitaciones + rng.normal(0, 0.25, n))      # millones de COP / mes
    return pd.DataFrame({"area_m2": area, "area_util_m2": area_util, "habitaciones": habitaciones,
                         "estrato": estrato, "antiguedad_anios": antiguedad, "distancia_centro_km": distancia,
                         "ruido_1": ruido[:, 0], "ruido_2": ruido[:, 1], "ruido_3": ruido[:, 2],
                         "arriendo_millones": arriendo})


df = generar_apartamentos()
X = df.drop(columns="arriendo_millones")
y = df["arriendo_millones"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
display(df.head())

---
## Reto 1: La regresión lineal por dentro y por fuera 📐

**Contexto:** una inmobiliaria quiere una primera estimación del arriendo. Antes de confiar en `LinearRegression`, vas a comprobar que entiendes lo que hace: resolver las **ecuaciones normales** $\hat{\boldsymbol\beta} = (X^\top X)^{-1} X^\top \mathbf{y}$.

**Enunciado**
1. Ajusta `ols = LinearRegression()` con `X_train, y_train`.
2. Construye la matriz de diseño `Xb` (una columna de unos seguida de `X_train`) y calcula `beta_np` resolviendo $(X_b^\top X_b)\,\boldsymbol\beta = X_b^\top \mathbf{y}$ con `np.linalg.solve` (no calcules la inversa).
3. Calcula en el conjunto de **prueba**: `rmse_ols` y `r2_ols` del modelo, y `rmse_base`, el RMSE de un `DummyRegressor()` (predice siempre la media de entrenamiento).

**Criterios de aceptación:** `beta_np` coincide con el intercepto y los coeficientes de `ols`; las métricas se recomputan correctamente; el modelo reduce el RMSE de la línea base en más del 40 %.

**Puntaje sugerido:** 20 puntos.

In [ ]:
# TODO: completa
ols = None         # LinearRegression ajustado con X_train, y_train
Xb = None          # matriz de diseño: columna de unos + X_train (shape: n x (p+1))
beta_np = None     # vector (p+1,) resuelto con np.linalg.solve; el elemento 0 es el intercepto

rmse_ols = None    # RMSE de `ols` en prueba
r2_ols = None      # R² de `ols` en prueba
rmse_base = None   # RMSE en prueba de DummyRegressor() ajustado con entrenamiento

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert isinstance(ols, LinearRegression) and hasattr(ols, "coef_"), "'ols' debe ser un LinearRegression ya ajustado (usa .fit)."
assert Xb is not None and Xb.shape == (len(X_train), X_train.shape[1] + 1) and np.allclose(Xb[:, 0], 1), "Xb debe tener una primera columna de unos seguida de las columnas de X_train."
assert beta_np is not None and np.shape(beta_np) == (X_train.shape[1] + 1,), "beta_np debe tener p+1 elementos (intercepto + un coeficiente por variable)."
assert np.isclose(beta_np[0], ols.intercept_, rtol=1e-5, atol=1e-6) and np.allclose(beta_np[1:], ols.coef_, rtol=1e-5, atol=1e-6), "La ecuación normal debe reproducir el intercepto y los coeficientes de LinearRegression."
_p = ols.predict(X_test)
assert rmse_ols is not None and abs(rmse_ols - np.sqrt(mean_squared_error(y_test, _p))) < 1e-9, "rmse_ols no coincide con la raíz del MSE de 'ols' en el conjunto de prueba."
assert r2_ols is not None and abs(r2_ols - r2_score(y_test, _p)) < 1e-9, "r2_ols no coincide con el R² de 'ols' en prueba."
assert rmse_base is not None and abs(rmse_base - y_test.std(ddof=0)) < 0.1, "rmse_base debe ser el RMSE de predecir siempre la media: ≈ la desviación estándar de y_test."
assert rmse_ols < 0.6 * rmse_base, "El modelo lineal debería reducir el RMSE de la línea base en más del 40 %."
print(f"✅ Reto 1 superado: RMSE = {rmse_ols:.3f} (base {rmse_base:.3f}) | R² = {r2_ols:.3f}")

---
## Reto 2: Variables gemelas y regularización Ridge 👯

**Contexto:** `area_m2` y `area_util_m2` miden casi lo mismo. Con variables tan correlacionadas los coeficientes de mínimos cuadrados se vuelven inestables. Lo diagnosticas con el **VIF**:
$$\mathrm{VIF}_j = \frac{1}{1 - R_j^2},$$
donde $R_j^2$ es el $R^2$ de regresar la columna $j$ sobre todas las demás. Valores por encima de 10 son señal de alarma.

**Enunciado**
1. Calcula `vif`: `pd.Series` con el VIF de cada columna de `X_train` (índice = nombre de la columna).
2. Guarda en `variable_redundante` el nombre de una de las dos columnas con VIF por encima de 10 (la de mayor VIF).
3. Ajusta `modelo_std = Pipeline([StandardScaler, LinearRegression])` y `ridge = Pipeline([StandardScaler, RidgeCV(alphas=np.logspace(-3, 3, 25))])` con el entrenamiento.
4. Calcula `norma_ols` y `norma_ridge` (norma euclídea de los coeficientes del último paso de cada pipeline, en unidades estandarizadas) y `r2_ridge` (R² de `ridge` en prueba).

**Criterios de aceptación:** solo las dos columnas gemelas superan VIF = 10 (las demás quedan por debajo de 5); Ridge **encoge** los coeficientes (`norma_ridge <= norma_ols`) sin perder capacidad predictiva ($R^2 > 0.78$).

**Puntaje sugerido:** 25 puntos.

In [ ]:
# Pista: para cada columna c, regresa X_train[c] sobre X_train.drop(columns=c) y usa VIF = 1 / (1 - R²)
vif = None                   # pd.Series
variable_redundante = None   # str

# TODO: arma y ajusta los dos pipelines (ambos con StandardScaler como primer paso)
modelo_std = None
ridge = None

norma_ols = None     # np.linalg.norm de los coeficientes de la regresión (último paso de modelo_std)
norma_ridge = None   # ídem para ridge
r2_ridge = None      # R² de ridge en prueba

In [ ]:
# Autoverificación del Reto 2 (no modificar)
assert isinstance(vif, pd.Series) and set(vif.index) == set(X_train.columns), "'vif' debe ser una pd.Series con una entrada por columna de X_train."
_gem = ["area_m2", "area_util_m2"]
assert (vif[_gem] > 10).all(), "Las dos columnas gemelas (area_m2 y area_util_m2) deben tener VIF > 10: revisa la fórmula 1 / (1 - R²)."
assert vif.drop(_gem).max() < 5, "Las demás columnas deberían tener VIF < 5: cada R² debe calcularse regresando la columna sobre TODAS las demás."
assert variable_redundante in _gem, "'variable_redundante' debe ser una de las dos columnas gemelas."
assert isinstance(modelo_std, Pipeline) and hasattr(modelo_std[-1], "coef_"), "'modelo_std' debe ser un Pipeline ajustado (StandardScaler + LinearRegression)."
assert isinstance(ridge, Pipeline) and hasattr(ridge[-1], "alpha_"), "'ridge' debe ser un Pipeline ajustado que termina en RidgeCV."
assert abs(norma_ols - np.linalg.norm(modelo_std[-1].coef_)) < 1e-9 and abs(norma_ridge - np.linalg.norm(ridge[-1].coef_)) < 1e-9, "Las normas deben calcularse con los coeficientes del último paso de cada pipeline."
assert norma_ridge <= norma_ols + 1e-12, "Ridge encoge los coeficientes: su norma no puede superar la de mínimos cuadrados."
assert r2_ridge is not None and abs(r2_ridge - ridge.score(X_test, y_test)) < 1e-9 and r2_ridge > 0.78, "r2_ridge debe ser el R² de 'ridge' en prueba y superar 0.78."
print(f"✅ Reto 2 superado: VIF gemelas = {vif[_gem].round(0).tolist()} | ‖β‖ OLS = {norma_ols:.3f} → Ridge = {norma_ridge:.3f}")

---
## Reto 3: De la regresión a la clasificación — apartamentos «premium» 🎲

**Contexto:** la inmobiliaria quiere una etiqueta comercial: un apartamento es **premium** si su arriendo supera el **percentil 75 del arriendo de entrenamiento**. Les preocupa *no dejar pasar* premiums, así que exigen una **sensibilidad (recall) de al menos 0.90**.

**Enunciado**
1. Crea `y_train_c` e `y_test_c` (0/1): 1 si el arriendo supera el percentil 75 de `y_train` (el mismo umbral para ambos conjuntos).
2. Ajusta `clf = Pipeline([StandardScaler, LogisticRegression(max_iter=1000)])` con `X_train, y_train_c` y calcula `auc_clf` en prueba.
3. Obtén `proba_cv`: probabilidades de la clase 1 sobre el **entrenamiento** con `cross_val_predict(..., cv=cv, method="predict_proba")[:, 1]` (así eliges el umbral sin tocar la prueba).
4. Recorre `np.linspace(0.05, 0.95, 91)` y guarda en `umbral` el **mayor** valor cuyo recall sobre `proba_cv` (predecir 1 si `proba_cv >= umbral`) sea $\ge 0.90$; guarda ese recall en `recall_umbral`.
5. `odds_ratios`: `pd.Series` con $e^{\beta_j}$ de cada variable (índice = columnas de `X_train`); al estar estandarizadas, es el efecto multiplicativo en los *odds* por **una desviación estándar**.

**Criterios de aceptación:** etiquetas bien construidas; AUC en prueba $> 0.90$; el umbral cumple el recall y es el mayor que lo hace; más estrato y menos antigüedad o distancia aumentan los *odds* de ser premium.

**Puntaje sugerido:** 25 puntos.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)   # no lo cambies

# TODO: completa
y_train_c = None     # np.ndarray / Serie 0-1 (premium) para entrenamiento
y_test_c = None      # idem para prueba, con el MISMO corte (percentil 75 de y_train)
clf = None           # Pipeline(StandardScaler, LogisticRegression(max_iter=1000)) ajustado con y_train_c
auc_clf = None       # AUC en prueba

proba_cv = None      # probabilidades de clase 1 vía cross_val_predict sobre el entrenamiento
umbral = None        # mayor umbral de la rejilla con recall >= 0.90 sobre proba_cv
recall_umbral = None # recall alcanzado con ese umbral
odds_ratios = None   # pd.Series: exp(coeficiente) por variable

In [ ]:
# Autoverificación del Reto 3 (no modificar)
_corte = np.percentile(y_train, 75)
assert y_train_c is not None and np.array_equal(np.asarray(y_train_c), (y_train > _corte).astype(int)), "y_train_c debe ser 1 si el arriendo supera el percentil 75 de y_train."
assert y_test_c is not None and np.array_equal(np.asarray(y_test_c), (y_test > _corte).astype(int)), "y_test_c debe usar el MISMO corte (percentil 75 de y_train), no el de la prueba."
assert isinstance(clf, Pipeline) and hasattr(clf[-1], "coef_"), "'clf' debe ser un Pipeline ajustado que termina en LogisticRegression."
assert auc_clf is not None and abs(auc_clf - roc_auc_score(y_test_c, clf.predict_proba(X_test)[:, 1])) < 1e-9 and auc_clf > 0.90, "auc_clf debe ser el AUC de 'clf' en prueba y superar 0.90."
_ref = cross_val_predict(clf, X_train, np.asarray(y_train_c), cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
assert proba_cv is not None and np.allclose(proba_cv, _ref, atol=1e-8), "proba_cv debe venir de cross_val_predict sobre el entrenamiento con el 'cv' dado y method='predict_proba' (columna de la clase 1)."
assert umbral is not None and recall_score(y_train_c, proba_cv >= umbral) >= 0.90, "Con tu umbral el recall sobre proba_cv es menor que 0.90."
assert recall_score(y_train_c, proba_cv >= umbral + 0.01) < 0.90 or umbral >= 0.95 - 1e-9, "Tu umbral no es el MAYOR de la rejilla que cumple recall >= 0.90: prueba subir el umbral."
assert recall_umbral is not None and abs(recall_umbral - recall_score(y_train_c, proba_cv >= umbral)) < 1e-9, "recall_umbral debe ser el recall con tu umbral sobre proba_cv."
assert isinstance(odds_ratios, pd.Series) and list(odds_ratios.index) == list(X_train.columns), "odds_ratios debe tener como índice las columnas de X_train."
assert odds_ratios["estrato"] > 1 and odds_ratios["antiguedad_anios"] < 1 and odds_ratios["distancia_centro_km"] < 1, "Un estrato mayor debe subir los odds de premium; más antigüedad y más distancia deben bajarlos (e^β > 1 sube, < 1 baja)."
print(f"✅ Reto 3 superado: AUC = {auc_clf:.3f} | umbral = {umbral:.2f} (recall {recall_umbral:.2f}) | OR(estrato) = {odds_ratios['estrato']:.2f}")

---
## Reto 4: Lasso, o cómo dejar fuera lo que sobra ✂️

**Contexto:** la penalización L1 puede llevar coeficientes **exactamente a cero**, haciendo selección de variables. Nuestras columnas `ruido_1`, `ruido_2` y `ruido_3` no tienen relación con el arriendo; ¿las descarta Lasso?

**Enunciado**
1. Ajusta `lasso = Pipeline([StandardScaler, LassoCV(cv=5, random_state=0)])` con `X_train, y_train`.
2. `coef_lasso`: `pd.Series` con sus coeficientes (índice = columnas de `X_train`).
3. `n_ceros`: cuántos coeficientes valen exactamente 0; `r2_lasso`: $R^2$ en prueba.

**Criterios de aceptación:** al menos una de las tres columnas de ruido queda en cero exacto; el $R^2$ de Lasso difiere en menos de 0.03 del de `ols` (Reto 1): se simplificó el modelo sin perder precisión.

**Puntaje sugerido:** 15 puntos.

In [ ]:
# TODO: completa
lasso = None        # Pipeline(StandardScaler, LassoCV(cv=5, random_state=0)) ajustado
coef_lasso = None   # pd.Series de coeficientes
n_ceros = None      # número de coeficientes exactamente 0
r2_lasso = None     # R² en prueba

In [ ]:
# Autoverificación del Reto 4 (no modificar)
assert isinstance(lasso, Pipeline) and hasattr(lasso[-1], "alpha_"), "'lasso' debe ser un Pipeline ajustado que termina en LassoCV."
assert isinstance(coef_lasso, pd.Series) and list(coef_lasso.index) == list(X_train.columns), "coef_lasso debe tener como índice las columnas de X_train."
assert np.allclose(coef_lasso.values, lasso[-1].coef_), "coef_lasso debe contener los coeficientes (coef_) del último paso del pipeline."
assert n_ceros == int((coef_lasso == 0).sum()), "n_ceros debe contar los coeficientes exactamente iguales a 0."
assert (coef_lasso[["ruido_1", "ruido_2", "ruido_3"]] == 0).sum() >= 1, "Lasso debería anular al menos una columna de ruido: ¿estandarizaste las variables antes de LassoCV?"
assert r2_lasso is not None and abs(r2_lasso - lasso.score(X_test, y_test)) < 1e-9, "r2_lasso debe ser el R² de 'lasso' en prueba."
assert abs(r2_lasso - r2_ols) < 0.03, "Lasso debería conservar casi el mismo R² que el modelo de mínimos cuadrados (diferencia < 0.03)."
print(f"✅ Reto 4 superado: Lasso anuló {n_ceros} coeficientes y conservó R² = {r2_lasso:.3f} (OLS: {r2_ols:.3f})")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- ¿Cuánto mejoró el modelo lineal frente a la línea base (RMSE y $R^2$) y qué significa el RMSE en pesos para la inmobiliaria?
- ¿Qué pasaba con `area_m2` y `area_util_m2`? ¿Qué hicieron Ridge y Lasso con ellas y por qué eso estabiliza la interpretación?
- Con tu umbral para recall $\ge 0.90$, ¿qué ganas y qué pierdes frente al umbral 0.5? (Piensa en falsos positivos.)
- Interpreta un *odds ratio* de tu tabla (por ejemplo el de `estrato`) en una frase para un gerente sin formación estadística.

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: OLS con scikit-learn, ecuación normal en NumPy y comparación con la línea base | 20 |
| Reto 2: VIF correcto y comparación OLS vs. Ridge con estandarización | 25 |
| Reto 3: etiqueta premium, regresión logística, umbral elegido sin tocar la prueba y *odds ratios* | 25 |
| Reto 4: Lasso, coeficientes nulos y comparación de $R^2$ | 15 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] Estandaricé las variables antes de Ridge, Lasso y regresión logística.
- [ ] El umbral del Reto 3 se eligió con validación cruzada del entrenamiento, no con la prueba.
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Machine Learning</i>
  </p>
</div>